# 00 - Review the Collaborator Data

**Purpose:** turn the received demand and map files into clear tables of substations, transmission routes and power stations for the existing-system model.

**Before running:**

- use the repository `.venv` kernel;
- place the expected folders under `data/0-incoming/energy/collaborator`, or set `MU_STAR_DATA_ROOT` to another data folder with the same structure;
- leave received files and filenames unchanged.

**What this notebook does:** reads the supported source files, writes cleaned tables under `data/1-processed/energy/collaborator`, counts the records found, maps their coverage and extracts the demand information in the workbook.

**What you need to do:** check the map and tables against your knowledge of the Mauritius system. Copy the generated template to `existing_generators.csv`, then add confirmed maximum output, fuel or technology, running cost, status and connected substation for each power station. The notebook does not estimate a station's output from its mapped size or assume that every unnamed polygon is a power station.

**Common adjustments:** you can use a different main data folder, change plot styles or update how clearly labelled source records are grouped. If a new source has different columns or file formats, update the reading code and record where the new data came from.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from mu_star_energy.intake import prepare_collaborator_data
from mu_star_energy.paths import incoming_energy_dir, processed_energy_dir, repo_root

REPO_ROOT = repo_root()
INPUT_DIR = incoming_energy_dir() / "collaborator"
OUTPUT_DIR = processed_energy_dir() / "collaborator"

prepared = prepare_collaborator_data(INPUT_DIR, OUTPUT_DIR)
prepared

## Power-station and network records found

The generated power-station register keeps the source location and leaves key technical values blank. Maximum output, fuel, running cost and connected substation must be checked against CEB reports or other agreed sources before the electricity supply model can run.

In [ ]:
substations = gpd.read_parquet(prepared.substations)
routes = gpd.read_parquet(prepared.transmission_routes)
generation_points = gpd.read_parquet(prepared.generation_points)
generation_areas = gpd.read_parquet(prepared.generation_areas)
generation_register = pd.read_csv(prepared.generation_register_template)
monthly_peak = pd.read_csv(prepared.monthly_peak_demand, index_col="year")
annual_demand = pd.read_csv(prepared.annual_sector_demand)

inventory = pd.Series({
    "substations found": len(substations),
    "mapped transmission routes found": len(routes),
    "mapped generation points found": len(generation_points),
    "mapped generation areas found": len(generation_areas),
    "named power-station records": len(generation_register),
    "records with maximum output": int(generation_register["capacity_mw"].notna().sum()),
    "records with running cost": int(generation_register["marginal_cost"].notna().sum()),
    "records with connected substation": int(generation_register["bus_id"].notna().sum()),
})
display(inventory.to_frame("count"))

display_register = generation_register[
    ["generator_id", "name", "asset_type", "capacity_mw", "marginal_cost", "bus_id", "status"]
].rename(columns={
    "generator_id": "power-station ID",
    "name": "name",
    "asset_type": "fuel or technology",
    "capacity_mw": "maximum output (MW)",
    "marginal_cost": "running cost (EUR/MWh)",
    "bus_id": "connected substation ID",
    "status": "review status",
})
display(display_register)

## Map of the received assets

Power-station areas are shown as transparent shapes above the transmission routes. Named sites are also shown as markers because many source polygons are too small to see at island scale.

In [ ]:
category_colors = {
    "thermal": "#8e24aa",
    "hydro": "#16a34a",
    "solar": "#c026d3",
    "wind": "#2563eb",
    "substation": "#f97316",
    "unspecified": "#9ca3af",
}
category_markers = {"thermal": "s", "hydro": "^", "solar": "v", "wind": "x", "unspecified": "D"}

fig, ax = plt.subplots(figsize=(10, 10))
routes.plot(ax=ax, color="#dc2626", linewidth=1.3, alpha=0.75, zorder=2)
for category, group in generation_areas.groupby("category"):
    alpha = 0.18 if category == "unspecified" else 0.4
    group.plot(
        ax=ax,
        color=category_colors.get(category, "#777777"),
        edgecolor="black" if category != "unspecified" else "none",
        linewidth=0.3,
        alpha=alpha,
        zorder=4,
    )
substations.plot(ax=ax, color="#ff8c00", edgecolor="black", markersize=42, zorder=6)
for category, group in generation_register.groupby("asset_type"):
    ax.scatter(
        group["lon"],
        group["lat"],
        c=category_colors.get(category, "#777777"),
        marker=category_markers.get(category, "D"),
        s=95,
        edgecolors="black" if category != "wind" else None,
        linewidths=1,
        label=category,
        zorder=8,
    )
ax.set_title("Collaborator transmission, substations and generation sites")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.legend(title="Named generation", fontsize="small")
plt.show()

## Demand information in the workbook

The workbook provides observed monthly system peaks and annual electricity use by customer group. It does not provide the hour-by-hour demand at each substation that the interruption model ultimately needs.

In [ ]:
display(monthly_peak.tail(8).style.format("{:.1f}"))
display(
    annual_demand.pivot(index="year", columns="category", values="demand_gwh")
    .tail(8)
    .style.format("{:.1f}")
)

## How to interpret and update the data

- Keep received source files unchanged under `data/0-incoming`.
- Do not estimate power-station output from polygon area.
- Do not treat unnamed mapped areas as confirmed power stations.
- Record the source and a short explanation for values added manually.
- Clear notebook outputs before committing if they contain private data or large figures.